# aiayn: Kaggle training (resumable)

Runs `scripts/train.py` for one experiment within a Kaggle session (docs/KAGGLE.md §7, ADR-013).

* **New run:** set `EXPERIMENT` and `RUN_NAME`, leave `RESUME_FROM = None`.
* **Continue a run** in a later session: add the previous notebook version's **output** as an input,
  then set `RESUME_FROM` to its run folder (`/kaggle/input/<output>/runs/<RUN_NAME>`).
* Use **Save Version → Save & Run All (Commit)** so `/kaggle/working/runs/` is kept as the output.

Settings: Accelerator **GPU T4**, Internet **ON** (code from GitHub, data downloaded with checksums).
Run the validation notebook first after any environment change.

In [ ]:
# 0. Parameters
GITHUB_URL = (
    "https://github.com/mohamed242006/Transformer-from-scratch-with-a-paper-reproduction.git"
)
GITHUB_BRANCH = "master"
EXPERIMENT = "configs/experiment/multi30k_tiny.yaml"
RUN_NAME = "multi30k_tiny_s1"
RESUME_FROM = None  # e.g. "/kaggle/input/<previous-output>/runs/multi30k_tiny_s1"
OVERRIDES = []  # e.g. ["train.seed=2"] (new runs) or ["train.max_epochs=40"] (resume)
WORK = "/kaggle/working"

In [ ]:
# 1. Code and dependencies (torch is never reinstalled)
import glob
import importlib.util
import os
import subprocess
import sys

REPO = f"{WORK}/aiayn"
if not os.path.exists(f"{REPO}/pyproject.toml"):
    clone = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_URL, REPO]
    subprocess.run(clone, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
missing = [
    s
    for m, s in {"sentencepiece": "sentencepiece>=0.2.0", "sacrebleu": "sacrebleu>=2.4"}.items()
    if importlib.util.find_spec(m) is None
]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code: git", head)

In [ ]:
# 2. Train (new run or resume). Exit code 3 = stopped on the session budget: resume next session.
run_dir = f"{WORK}/runs/{RUN_NAME}"
cmd = [sys.executable, "-u", "scripts/train.py", "--run-dir", run_dir]
if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM, "--prepare-data"]  # fresh session: re-download data
else:  # data is prepared under the repo (data/), i.e. inside /kaggle/working
    cmd += ["--experiment", EXPERIMENT, "--prepare-data"]
for item in OVERRIDES:
    cmd += ["--override", item]
print("$", " ".join(cmd))
TRAIN_EXIT = subprocess.run(cmd).returncode
print("exit code:", TRAIN_EXIT, "(0 = completed, 3 = resumable)")

In [ ]:
# 3. Summary of the run directory (kept as notebook output)
import json

with open(f"{run_dir}/metadata.json") as fh:
    meta = json.load(fh)
print("best valid NLL:", meta.get("best_valid_nll"), "at update", meta.get("best_update"))
for session in meta["sessions"]:
    print(session)
with open(f"{run_dir}/metrics.jsonl") as fh:
    records = [json.loads(line) for line in fh]
valid = [r for r in records if r["split"] == "valid"]
for r in valid[-5:]:
    print(f"update {r['step']:>6}  nll {r['nll']:.4f}  ppl {r['ppl']:.2f}  acc {r['accuracy']:.4f}")
assert TRAIN_EXIT in (0, 3), "training failed: see the log above"

In [ ]:
# 4. Evaluate on the test set: sacreBLEU (cased, primary), lowercased BLEU, chrF (ADR-006)
import json as _json

if TRAIN_EXIT == 0:
    for extra in (
        ["--checkpoint", "best"],
        ["--checkpoint", "averaged"],
        ["--checkpoint", "best", "--greedy"],
    ):
        subprocess.run(
            [
                sys.executable,
                "scripts/evaluate.py",
                "--run-dir",
                run_dir,
                "--split",
                "test",
                *extra,
            ],
            check=True,
        )
    for path in sorted(glob.glob(f"{run_dir}/eval/test.*.json")):
        with open(path) as fh:
            r = _json.load(fh)
        name = os.path.basename(path)
        print(f"{name:<32} BLEU {r['bleu']:6.2f}  lc-BLEU {r['bleu_lowercase']:6.2f}", end="")
        print(f"  chrF {r['chrf']:6.2f}  ({r['decode_seconds']} s)")
else:
    print("training not finished yet: resume in the next session, evaluate afterwards")